# Unit 5: Training with Weak Targets: Advanced

[Course overview](00_overview.ipynb)

<ul>
<li><a href="#learn" style="color:#E65100;"><strong>Overview and Learning Objectives</strong></a></li>
<li><a href="#compact" style="color:black;">Compact Weak Targets</a></li>
<li><a href="#steps" style="color:black;">Step Sizes and Weights</a></li>
<li><a href="#modified_targets" style="color:#1B5E20;">Background: Modified Targets</a></li>
<li><a href="#weak_training" style="color:black;">Training with Compact Targets</a></li>
<li><a href="#exercise_parameters" style="color:#006064;">Exercise 1: Standard SDTW Parameters</a></li>
<li><a href="#loss_offset" style="color:black;">Comparing Losses across Temperatures</a></li>
<li><a href="#exercise_offset" style="color:#006064;">Exercise 2: Path-Count Offset</a></li>
<li><a href="#predictions" style="color:black;">Inspecting Predictions</a></li>
<li><a href="#shift" style="color:black;">Temporal Shifts</a></li>
</ul>

<a id="learn"></a>

<div style="
  background-color: #FFE0B2A0;
  border: 1px solid #FFC671A0;
  padding-top: 10px;
  padding-bottom: 10px;
  padding-left: 20px;
  padding-right: 20px;
  margin-top: 10px;
  margin-bottom: 10px;
">

<h2 style="color:#E65100; margin-top:0;">Overview and Learning Objectives</h2>

<p style="color:#E65100;">In this unit, we train directly from compact weak targets and examine
    how step sizes, step weights, and temperature shape the alignment. We inspect
    modified targets, distinguish loss offsets from prediction quality, and
    diagnose systematic shifts that matter especially for onset detection.</p>

</div>

In [ ]:
from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from torch.utils.data import DataLoader
from ddtw import SDTW

from dataset_multipitch import MultipitchDataset, collate_multipitch
from model import Transcription_CNN

# --- Figure display and style ---
%config InlineBackend.figure_formats = ['svg']
plt.rcParams.update({"font.size": 8, "axes.titlesize": 9})

In [ ]:
# config
torch.manual_seed(1)
device = torch.device("cpu")

max_polyphony = 3  # maximum number of simultaneous onsets in the dataset
# activation function for the output layer of the model
output_activation = "sigmoid"
freqs = (30, 35, 40, 45)  # frequencies (Hz) of the sinusoids in the dataset

In [ ]:
# data
train_data = MultipitchDataset(
    size=256, seed=0, noise_std=0.01, max_polyphony=max_polyphony, freqs=freqs
)
valid_data = MultipitchDataset(
    size=32, seed=1, noise_std=0.01, max_polyphony=max_polyphony, freqs=freqs
)
train_loader = DataLoader(
    train_data,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_multipitch,
    num_workers=0,
)
valid_loader = DataLoader(
    valid_data, batch_size=16, collate_fn=collate_multipitch, num_workers=0
)
class_names = train_data.class_names

<a id="compact"></a>

## Compact Weak Targets


We use the same data and model as in the previous notebook. This time
the target consists of one vector per state, and we completely
discard duration information. As this results in different sequence lengths of the
weak targets, we have to keep the cached sequence lengths when batching and pass them
to SDTW as `list_M=target_lengths_frames`.

In [ ]:
# plot a sample waveform and its corresponding strong and weak targets

(
    test_waveform,
    test_strong_targets_frames,
    test_weak_targets_frames,
    _,
    _,
) = valid_data[0]

fig, ax = plt.subplots(3, 1, figsize=(6.2, 4.8))

im0 = ax[0].plot(np.arange(0, 500, 1), test_waveform[0].numpy(), color="k")
ax[0].set(
    xlim=[0, 500],
    title="Input waveform",
    ylabel="Amplitude",
    xlabel="Frame index",
)

im1 = ax[1].imshow(
    test_strong_targets_frames.numpy().T,
    aspect="auto",
    origin="lower",
    interpolation="None",
    cmap="gray_r",
    extent=[0, 500, -0.5, len(class_names) - 0.5],
)
ax[1].set(
    xlim=[0, 500],
    title="Strong targets (frames)",
    ylabel="Class",
    yticks=np.arange(len(class_names)),
    yticklabels=class_names,
    xlabel="Frame index",
)
plt.colorbar(im1, ax=ax[1], label="Amplitude")
cbar = plt.colorbar(im1, ax=ax[0], label=None)
fig.delaxes(cbar.ax)

im2 = ax[2].imshow(
    test_weak_targets_frames.numpy().T,
    aspect="auto",
    origin="lower",
    interpolation="None",
    cmap="gray_r",
)
ax[2].set(
    title="Weak targets (frames)",
    ylabel="Class",
    yticks=np.arange(len(class_names)),
    yticklabels=class_names,
    xticks=np.arange(0, test_weak_targets_frames.shape[0], 1),
    xlabel="Weak target index",
)
plt.colorbar(im2, ax=ax[2], label="Amplitude")

fig.tight_layout()
plt.show()

<a id="batching"></a>

<div style="
  background-color: #C8E6C9A0;
  border: 1px solid #81C784A0;
  padding: 10px 20px;
  margin: 10px 0;
">

<h2 style="color:#1B5E20; margin-top:0;">Background: Variable-Length Targets and Batching</h2>

The collation function caches each original weak-target length before
padding sequences into a batch. These lengths mus be passed to the SDTW
forward and backward recurrence to evaluate them on the valid region, excluding padding.
These lengths are passed to the SDTW cost as <code>list_M=target_lengths_frames</code>
and <code>list_N</code>. 

</div>

<a id="steps"></a>

## Step Sizes and Weights


The step set specifies which correspondences are valid:

| Step $(\Delta n,\Delta m)$ | Meaning | Local-cost multiplier |
| --- | --- | --- |
| $(1,0)$ | Advance prediction time, retain target state | 0.1 |
| $(1,1)$ | Advance both sequences | 1.0 |
| $(0,1)$ | Advance target state, reuse prediction frame | 1.0 |

`step_sizes` controls allowed increments; `global_step_weights` multiplies
their local costs in the same order. Our experiment retains all
three steps and uses `[0.1, 1.0, 1.0]`. The smaller first multiplier makes
staying in a target state cheaper, which can help when $N\gg M$ but does
not guarantee a good alignment.

<a id="modified_targets"></a>

<div style="
  background-color: #C8E6C9A0;
  border: 1px solid #81C784A0;
  padding: 10px 20px;
  margin: 10px 0;
">

<h2 style="color:#1B5E20; margin-top:0;">Background: Modified Targets and the Prediction Gradient</h2>

Let $\mathbf{H}=\partial L_{\rm SDTW}/\partial \mathbf{C}$. With unit step weights, $\mathbf{H}$ it is equal to the expected alignment $\mathbf{E}$;
for nonuniform step weights it includes the local-cost multipliers.
Define $h_n=\sum_m \mathbf{H}(n,m)$ and $$\widetilde{Y}(n,k)=\frac{\sum_m \mathbf{H}(n,m)Y(m,k)}{h_n}.$$
This moves the weak targets to the prediction timeline. Row normalization
makes each modified target a convex combination, retaining its meaning as
an activity value between zero and one. This reformulation allows us to interpret training with weak targets $Y$ and SDTW
as equivalently training with strong targets $\widetilde{Y}$ and an element-wise loss function. Note
that this is only meant to analyze what the network "sees" during training, and is not actually used as a training 
objective.

For further details, see 
>Johannes Zeitler and Meinard Müller. Reformulating soft dynamic time warping: Insights into target artifacts and prediction 
> quality. In Proceedings of the International Society for Music Information Retrieval Conference (ISMIR), 2025 pages
> 127–133, Daejeon, South Korea. https://doi.org/10.5281/ZENODO.17706349.

</div>

## Training with weak targets and non-uniform step weights

In [ ]:
# initialize SDTW

sdtw_loss = SDTW(
    gamma=10.0,
    step_sizes=[[1, 0], [1, 1], [0, 1]],
    cost_function="BCE",
    store_debug=True,
    global_step_weights=[0.1, 1.0, 1.0],
    normalization="N",
    backend="cpu_numba",
)

# initialize model
model_sdtw = Transcription_CNN(
    num_classes=train_data.num_classes, activation=output_activation
).to(device)


optimizer = torch.optim.Adam(model_sdtw.parameters(), lr=0.001)
train_losses, valid_losses, precisions, recalls, fmeasures, gammas = (
    [],
    [],
    [],
    [],
    [],
    [],
)
training_start = perf_counter()

for epoch in range(100):
    # train step
    model_sdtw.train()
    total_loss = 0.0
    gammas.append(sdtw_loss.gamma.item())
    for (
        waveforms,
        strong_targets_frames,
        weak_targets_frames,
        strong_targets_onsets,
        weak_targets_onsets,
        target_lengths_frames,
        target_lengths_onsets,
    ) in train_loader:
        optimizer.zero_grad()
        predictions = model_sdtw(waveforms)

        # compute SDTW loss between predictions and unaligned weak targets
        loss = sdtw_loss(
            X=predictions, Y=weak_targets_frames, list_M=target_lengths_frames
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(waveforms)
    train_losses.append(total_loss / len(train_data))

    # validation step
    # model_sdtw.eval()
    total_loss = 0.0
    # with torch.no_grad():
    p_ = []
    r_ = []
    f_ = []
    for (
        waveforms,
        strong_targets_frames,
        weak_targets_frames,
        strong_targets_onsets,
        weak_targets_onsets,
        target_lengths_frames,
        target_lengths_onsets,
    ) in valid_loader:
        predictions = model_sdtw(waveforms)

        loss = sdtw_loss(
            X=predictions, Y=weak_targets_frames, list_M=target_lengths_frames
        )
        loss.backward()
        total_loss += loss.item() * len(waveforms)

        # calculate precision, recall, and F1 score for each batch
        pred_thresh = (predictions > 0.5) * 1.0
        TP = (strong_targets_frames * pred_thresh).sum()
        FP = ((1 - strong_targets_frames) * pred_thresh).sum()
        FN = (strong_targets_frames * (1 - pred_thresh)).sum()
        P = TP / (TP + FP)
        R = TP / (TP + FN)
        F = 2 * P * R / (P + R)
        p_.append(P.item())
        r_.append(R.item())
        f_.append(F.item())

    valid_losses.append(total_loss / len(valid_data))

    # print stats every 10 epochs
    elapsed = perf_counter() - training_start
    if epoch == 0 or (epoch + 1) % 10 == 0:
        print(
            (
                "Epoch %3d | train loss %.5f | val loss %.4f | "
                "val P %.4f | val R %.4f | val F1 %.4f | "
                "elapsed time %.1f s"
            )
            % (
                epoch + 1,
                train_losses[-1],
                valid_losses[-1],
                np.mean(p_),
                np.mean(r_),
                np.mean(f_),
                elapsed,
            )
        )
        precisions.append(np.mean(p_))
        recalls.append(np.mean(r_))
        fmeasures.append(np.mean(f_))

        # ## plot the E matrix, the modified targets, and the strong targets for
        # a few samples
        n_visualizations = 4
        fig, ax = plt.subplots(3, n_visualizations, figsize=(6.2, 3.2))
        for i in range(n_visualizations):
            E = sdtw_loss.core.E_matrix[i]
            H = sdtw_loss.core.H_matrix[i]
            Y = weak_targets_frames[i]
            h = H.sum(axis=1, keepdims=True)
            H_tilde = H / h
            Y_mod = H_tilde @ Y
            ax[0, i].imshow(
                E.detach().numpy().T,
                aspect="auto",
                origin="lower",
                cmap="gray_r",
                interpolation="None",
            )
            ax[1, i].imshow(
                Y_mod.detach().numpy().T,
                aspect="auto",
                origin="lower",
                cmap="gray_r",
                interpolation="None",
            )
            ax[2, i].imshow(
                strong_targets_frames[i].detach().numpy().T,
                aspect="auto",
                origin="lower",
                cmap="gray_r",
                interpolation="None",
            )
        ax[0, 0].set_ylabel("E matrix")
        ax[1, 0].set_ylabel("Mod. targets")
        ax[2, 0].set_ylabel("Strong targets")
        ax[2, 0].set_xlabel("Time frames")
        fig.tight_layout()
        plt.show()
        ###################################################################

    # do a gamma annealing every 20 epochs
    if (epoch + 1) % 20 == 0:
        sdtw_loss.gamma *= 0.5
        print("Gamma updated to %.2f" % sdtw_loss.gamma)

# plot training and validation loss curves
fig, ax = plt.subplots(1, 2, figsize=(6.2, 2.2))
ax[0].plot(np.arange(1, len(train_losses) + 1), train_losses, label="train")
ax[0].plot(
    np.arange(1, len(valid_losses) + 1), valid_losses, label="validation"
)
ax[0].set(
    xlabel="Epoch", ylabel="Mean loss", title="Training and validation loss"
)
ax[0].legend()
ax[0].grid()

ax[1].plot(np.arange(1, len(precisions) + 1), precisions, label="precision")
ax[1].plot(np.arange(1, len(recalls) + 1), recalls, label="recall")
ax[1].plot(np.arange(1, len(fmeasures) + 1), fmeasures, label="F1 score")
ax[1].set(
    xlabel="Epoch", ylabel="Score", title="Precision, recall, and F1 score"
)
ax[1].legend()
ax[1].grid()
plt.show()

<a id="exercise_parameters"></a>

<div style="
  background-color: #B2EBF290;
  border: 1px solid #4DD0E190;
  padding-top: 10px;
  padding-bottom: 10px;
  padding-left: 20px;
  padding-right: 20px;
  margin-top: 10px;
  margin-bottom: 10px;
">

<strong style="color:#006064; display:block; break-after:avoid; page-break-after:avoid;">Exercise 1: Training with Standard SDTW Parameters</strong>

<ol style="color:#006064;">
<li>Repeat this compact-target experiment with unit step weights, keeping the gamma schedule. Then try fixed gamma = 1 and fixed gamma = 10. Change one setting at a time.</li>
<li>Compare frame F1, soft alignments, and modified targets. Identify collapse, blurring, or timing errors if they occur.</li>
</ol>

</div>

### Solution: Training with Standard SDTW Parameters

Use the function below for three unit-weight experiments: scheduled gamma
starting at 10, fixed gamma 1, and fixed gamma 10. Compare their outputs
with the weighted main experiment. The same initialization seed is reset
for each run. Each call uses a fresh CNN, optimizer, and SDTW object.

Scheduled smoothing can broadly distribute early gradients and sharpen them later.
A small fixed gamma can lock onto an initially poor path, while a large
fixed gamma can retain blurred modified targets. Unit weights change path
preferences relative to the weighted baseline. These are some common failure
mechanisms of SDTW. We report observed F1 and visually inspect predictions
for each run. Due to the SDTW loss definition, raw loss values across different 
objectives are not directly comparable.

The complete solution defines the experiments but leaves the additional
training runs disabled.

In [ ]:
def solution_standard_sdtw(initial_gamma, anneal, step_weights):
    torch.manual_seed(1)
    # initialize SDTW

    sdtw_loss = SDTW(
        gamma=initial_gamma,
        step_sizes=[[1, 0], [1, 1], [0, 1]],
        cost_function="BCE",
        store_debug=True,
        global_step_weights=step_weights,
        normalization="N",
        backend="cpu_numba",
    )

    # initialize model
    model_sdtw = Transcription_CNN(
        num_classes=train_data.num_classes, activation=output_activation
    ).to(device)

    optimizer = torch.optim.Adam(model_sdtw.parameters(), lr=0.001)
    train_losses, valid_losses, precisions, recalls, fmeasures, gammas = (
        [],
        [],
        [],
        [],
        [],
        [],
    )
    training_start = perf_counter()

    for epoch in range(100):
        # train step
        model_sdtw.train()
        total_loss = 0.0
        gammas.append(sdtw_loss.gamma.item())
        for (
            waveforms,
            strong_targets_frames,
            weak_targets_frames,
            strong_targets_onsets,
            weak_targets_onsets,
            target_lengths_frames,
            target_lengths_onsets,
        ) in train_loader:
            optimizer.zero_grad()
            predictions = model_sdtw(waveforms)

            # compute SDTW loss between predictions and unaligned weak targets
            loss = sdtw_loss(
                X=predictions,
                Y=weak_targets_frames,
                list_M=target_lengths_frames,
            )

            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(waveforms)
        train_losses.append(total_loss / len(train_data))

        # validation step
        # model_sdtw.eval()
        total_loss = 0.0
        # with torch.no_grad():
        p_ = []
        r_ = []
        f_ = []
        for (
            waveforms,
            strong_targets_frames,
            weak_targets_frames,
            strong_targets_onsets,
            weak_targets_onsets,
            target_lengths_frames,
            target_lengths_onsets,
        ) in valid_loader:
            predictions = model_sdtw(waveforms)

            loss = sdtw_loss(
                X=predictions,
                Y=weak_targets_frames,
                list_M=target_lengths_frames,
            )
            loss.backward()
            total_loss += loss.item() * len(waveforms)

            # calculate precision, recall, and F1 score for each batch
            pred_thresh = (predictions > 0.5) * 1.0
            TP = (strong_targets_frames * pred_thresh).sum()
            FP = ((1 - strong_targets_frames) * pred_thresh).sum()
            FN = (strong_targets_frames * (1 - pred_thresh)).sum()
            P = TP / (TP + FP)
            R = TP / (TP + FN)
            F = 2 * P * R / (P + R)
            p_.append(P.item())
            r_.append(R.item())
            f_.append(F.item())

        valid_losses.append(total_loss / len(valid_data))

        # print stats every 10 epochs
        elapsed = perf_counter() - training_start
        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(
                (
                    "Epoch %3d | train loss %.5f | val loss %.4f | "
                    "val P %.4f | val R %.4f | val F1 %.4f | "
                    "elapsed time %.1f s"
                )
                % (
                    epoch + 1,
                    train_losses[-1],
                    valid_losses[-1],
                    np.mean(p_),
                    np.mean(r_),
                    np.mean(f_),
                    elapsed,
                )
            )
            precisions.append(np.mean(p_))
            recalls.append(np.mean(r_))
            fmeasures.append(np.mean(f_))

            # ## plot the E matrix, the modified targets, and the strong targets
            # for a few samples
            n_visualizations = 4
            fig, ax = plt.subplots(3, n_visualizations, figsize=(6.2, 3.2))
            for i in range(n_visualizations):
                E = sdtw_loss.core.E_matrix[i]
                H = sdtw_loss.core.H_matrix[i]
                Y = weak_targets_frames[i]
                h = H.sum(axis=1, keepdims=True)
                H_tilde = H / h
                Y_mod = H_tilde @ Y
                ax[0, i].imshow(
                    E.detach().numpy().T,
                    aspect="auto",
                    origin="lower",
                    cmap="gray_r",
                    interpolation="None",
                )
                ax[1, i].imshow(
                    Y_mod.detach().numpy().T,
                    aspect="auto",
                    origin="lower",
                    cmap="gray_r",
                    interpolation="None",
                )
                ax[2, i].imshow(
                    strong_targets_frames[i].detach().numpy().T,
                    aspect="auto",
                    origin="lower",
                    cmap="gray_r",
                    interpolation="None",
                )
            ax[0, 0].set_ylabel("E matrix")
            ax[1, 0].set_ylabel("Mod. targets")
            ax[2, 0].set_ylabel("Strong targets")
            ax[2, 0].set_xlabel("Time frames")
            fig.tight_layout()
            plt.show()
            ###################################################################

        # do a gamma annealing every 20 epochs
        if anneal and (epoch + 1) % 20 == 0:
            sdtw_loss.gamma *= 0.5
            print("Gamma updated to %.2f" % sdtw_loss.gamma)

    # plot training and validation loss curves
    fig, ax = plt.subplots(1, 2, figsize=(6.2, 2.2))
    ax[0].plot(
        np.arange(1, len(train_losses) + 1), train_losses, label="train"
    )
    ax[0].plot(
        np.arange(1, len(valid_losses) + 1), valid_losses, label="validation"
    )
    ax[0].set(
        xlabel="Epoch",
        ylabel="Mean loss",
        title="Training and validation loss",
    )
    ax[0].legend()
    ax[0].grid()

    ax[1].plot(
        np.arange(1, len(precisions) + 1), precisions, label="precision"
    )
    ax[1].plot(np.arange(1, len(recalls) + 1), recalls, label="recall")
    ax[1].plot(np.arange(1, len(fmeasures) + 1), fmeasures, label="F1 score")
    ax[1].set(
        xlabel="Epoch", ylabel="Score", title="Precision, recall, and F1 score"
    )
    ax[1].legend()
    ax[1].grid()
    plt.show()
    return {"model": model_sdtw, "f1": fmeasures, "gamma": gammas}

In [ ]:
RUN_PARAMETER_EXERCISES = False
if RUN_PARAMETER_EXERCISES:
    scheduled = solution_standard_sdtw(10.0, True, [1.0, 1.0, 1.0])
    fixed_one = solution_standard_sdtw(1.0, False, [1.0, 1.0, 1.0])
    fixed_ten = solution_standard_sdtw(10.0, False, [1.0, 1.0, 1.0])

<a id="loss_offset"></a>

## Comparing Losses across Temperatures


For path costs $c_\mathbf{A}$, the softmin bound gives

$$\min_\mathbf{A} c_\mathbf{A}-\gamma\log K\leq L_{\rm SDTW}\leq\min_\mathbf{A} c_\mathbf{A},$$

where $K=|\mathcal{A}|$ counts valid paths. For the three standard steps
and endpoint constraints this is the Delannoy number .
A changing gamma therefore changes the scale of the loss even with fixed
predictions. Adding $\gamma\log K/N$ removes this
negative offset.

<a id="exercise_offset"></a>

<div style="
  background-color: #B2EBF290;
  border: 1px solid #4DD0E190;
  padding-top: 10px;
  padding-bottom: 10px;
  padding-left: 20px;
  padding-right: 20px;
  margin-top: 10px;
  margin-bottom: 10px;
">

<strong style="color:#006064; display:block; break-after:avoid; page-break-after:avoid;">Exercise 2: Compensating for the Path-Count Offset</strong>

<ol style="color:#006064;">
<li>Add gamma × log(D) / N to each epoch's normalized losses. For an illustrative plot use N = 500 and D = 8 × 10<sup>39</sup>; actual path counts vary with each weak-target length.</li>
<li>Inspect the jumps at gamma updates and explain why they may remain.</li>
<li>Compare the trend with strong-target BCE, accounting for their different reductions and objectives.</li>
</ol>

</div>

### Solution: Compensating for the Path-Count Offset


In [ ]:
train_l = np.array(train_losses)
val_l = np.array(valid_losses)
epoch_gammas = np.array(gammas)
n_paths = 8e39  # Illustrative constant, not a measured dataset path count.
offset = epoch_gammas * np.log(n_paths) / 500
fig, ax = plt.subplots(figsize=(5.2, 2.6), layout="tight")
epochs = np.arange(1, len(train_l) + 1)
ax.plot(epochs, train_l + offset, label="train + offset")
ax.plot(epochs, val_l + offset, label="validation + offset")
ax.set(
    xlabel="Epoch", ylabel="Corrected loss", title="Negative offset correction"
)
ax.grid()
ax.legend()
plt.show()

<a id="predictions"></a>

## Inspecting Predictions

Black denotes TP, red FP, cyan FN, and white TN.

In [ ]:
(
    test_waveform,
    test_strong_targets_frames,
    test_weak_targets_frames,
    _,
    _,
) = valid_data[0]
pred_weak_sdtw = model_sdtw(test_waveform[None])[0]

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(5.2, 5), squeeze=False)

ax[0, 0].imshow(
    test_strong_targets_frames.numpy().T,
    aspect="auto",
    origin="lower",
    interpolation="None",
    cmap="gray_r",
    clim=[0, 1],
)
ax[1, 0].imshow(
    pred_weak_sdtw.detach().numpy().T,
    aspect="auto",
    origin="lower",
    interpolation="None",
    cmap="gray_r",
    clim=[0, 1],
)

# thresholded predictions
pred_weak_thresh = (pred_weak_sdtw > 0.5) * 1.0
TP = test_strong_targets_frames * pred_weak_thresh
FP = (1 - test_strong_targets_frames) * pred_weak_thresh
FN = test_strong_targets_frames * (1 - pred_weak_thresh)
TN = (1 - test_strong_targets_frames) * (1 - pred_weak_thresh)
pred_RGB_sdtw = np.zeros((500, len(class_names), 3))
pred_RGB_sdtw[TP == 1] = [0, 0, 0]
pred_RGB_sdtw[FP == 1] = [1, 0, 0]
pred_RGB_sdtw[FN == 1] = [0, 1, 1]
pred_RGB_sdtw[TN == 1] = [1, 1, 1]
ax[2, 0].imshow(
    pred_RGB_sdtw.swapaxes(0, 1), aspect="auto", interpolation="None"
)

for d in range(len(class_names)):
    ax[3, 0].plot(np.arange(500), np.zeros(500) + d, color="grey", linewidth=2)
    ax[3, 0].plot(
        np.arange(500),
        pred_weak_sdtw.detach().numpy()[:, d] + d,
        color=f"C{d}",
        label=class_names[d],
        linewidth=1,
    )
    ax[3, 0].plot(
        np.arange(500),
        test_strong_targets_frames[:, d] + d,
        color=f"C{d}",
        linestyle="--",
    )
    ax[3, 0].grid()
ax[3, 0].plot(np.arange(500), np.zeros(500) + d + 1, color="grey", linewidth=2)
ax[3, 0].set(xlim=[-0.5, 499.5])


ax[1, 0].set(ylabel="Probabilities")
ax[2, 0].set(ylabel="thresh. pred.")
ax[3, 0].set(ylabel="signals", xlabel="Frame index")

ax[0, 0].set(title="Compact weak targets and SDTW", ylabel="Strong targets")
ax[3, 0].set(xlabel="Frame index")

red_patch = mpatches.Patch(color="red", label="FP")
cyan_patch = mpatches.Patch(color="cyan", label="FN")
black_patch = mpatches.Patch(color="black", label="TP")

ax[2, 0].legend(
    handles=[black_patch, red_patch, cyan_patch], loc="upper right"
)

fig.tight_layout()
plt.show()

<a id="shift"></a>

## Temporal Shifts 

A compact target sequence constrains target order but does not specify
absolute durations. A consistently shifted prediction can still
align cheaply by stretching states near the boundaries, and SDTW
is inherently invariant to such shifts. This matters
especially for onset detection: narrow peaks may miss the onset tolerance
even when sustained activity remains mostly correct. 

We therefore analyze the peak position of the cross-correlation between
predictions and strong targets to investigate such a global shift.

In [ ]:
preds = pred_weak_sdtw.detach()
#binary = (pred_weak_sdtw.detach() > 0.5).float()
reference = test_strong_targets_frames
lags = np.arange(-50, 51)
correlations = []
for lag in lags:
    if lag > 0:
        score = (preds[lag:] * reference[:-lag]).sum()
    elif lag < 0:
        score = (preds[:lag] * reference[-lag:]).sum()
    else:
        score = (preds * reference).sum()
    correlations.append(score.item())
best_lag = int(lags[np.argmax(correlations)])
print("Peak lag (positive = late prediction):", best_lag, "frames")
fig, ax = plt.subplots(figsize=(5.2, 2.2), layout="tight")
ax.plot(lags, correlations, label="weak SDTW")
ax.axvline(0, color="gray", linestyle="--")
ax.set(
    xlabel="Lag (frames)",
    ylabel="Activity overlap",
    title="Cross-correlation with strong targets",
)
ax.grid()
ax.legend()
plt.show()